<a href="https://colab.research.google.com/github/ced-sys/AI-N-ML/blob/main/Barbados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import pandas as pd
from pathlib import Path

In [3]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

Mounted at /content/drive


In [4]:
DATA_DIR='/content/drive/MyDrive/Barbados'

print(f"Data directory: {DATA_DIR}")

Data directory: /content/drive/MyDrive/Barbados


In [5]:
if os.path.exists(DATA_DIR):
    contents=os.listdir(DATA_DIR)
    print(f"\nContents of Barbados folder:")

    folders=[]
    files=[]

    for item in sorted(contents):
      item_path=os.path.join(DATA_DIR, item)
      if os.path.isdir(item_path):
        folders.append(item)

        try:
          video_count=len([f for f in os.listdir(item_path) if f.endswith('.mp4')])
          print(f" {item:20s} ({video_count} videos)")
        except:
          print(f"{item}")
      else:
        files.append(item)
        file_size=os.path.getsize(item_path)/(1024*1024)
        print(f"{item:20s} ({file_size:.2f} MB)")

    print(f"\nSummary")
    print(f" -{len(folders)} folders")
    print(f" -{len(files)} files")


Contents of Barbados folder:
TestInputSegments.csv (0.82 MB)
Train(1).csv         (4.87 MB)
 normanniles1         (4633 videos)
 normanniles2         (4633 videos)
 normanniles3         (4633 videos)
 normanniles4         (4633 videos)
 traffic_output       (0 videos)

Summary
 -5 folders
 -2 files


In [6]:
train_csv_paths=[os.path.join(DATA_DIR, 'Train(1).csv')]

TRAIN_CSV=None
for path in train_csv_paths:
    if os.path.exists(path):
        TRAIN_CSV=path
        break

if TRAIN_CSV:
    print(f"Training CSV found: {os.path.basename(TRAIN_CSV)}")

    train_df=pd.read_csv(TRAIN_CSV)
    print(f" -Shape: {train_df.shape}")
    print(f" -Columns: {list(train_df.columns)}")
    print(f"\n First few rows:")
    print(train_df.head(3))
else:
    print("Training CSV not found!")
    print("Looking for: Train(1).csv")

test_csv_paths=[
    os.path.join(DATA_DIR, 'TestInputSegments.csv')
]

TEST_CSV = None
for path in test_csv_paths:
    if os.path.exists(path):
        TEST_CSV = path
        break

if TEST_CSV:
    print(f"\nTest CSV found: {os.path.basename(TEST_CSV)}")

    test_df=pd.read_csv(TEST_CSV)
    print(f" -Shape: {test_df.shape}")
    print(f" -Columns: {list(test_df.columns)}")
    print(f"\n First few rows:")
    print(test_df.head(3))
else:
    print("\nTest CSV not found!")

Training CSV found: Train(1).csv
 -Shape: (16076, 14)
 -Columns: ['responseId', 'view_label', 'ID_enter', 'ID_exit', 'videos', 'video_time', 'datetimestamp_start', 'datetimestamp_end', 'date', 'signaling', 'congestion_enter_rating', 'congestion_exit_rating', 'time_segment_id', 'cycle_phase']

 First few rows:
                responseId       view_label  \
0  zYkHaeOdB7XOnvgP3YW5kQs  Norman Niles #1   
1  NYsHaeCRLq-vnvgPjoXZqA0  Norman Niles #1   
2  A40HaYT8KNm7nvgPq8e12AU  Norman Niles #1   

                                            ID_enter  \
0  time_segment_0_Norman Niles #1_congestion_ente...   
1  time_segment_1_Norman Niles #1_congestion_ente...   
2  time_segment_2_Norman Niles #1_congestion_ente...   

                                             ID_exit  \
0  time_segment_0_Norman Niles #1_congestion_exit...   
1  time_segment_1_Norman Niles #1_congestion_exit...   
2  time_segment_2_Norman Niles #1_congestion_exit...   

                                              vide

In [7]:
WORK_DIR="/content/Barbados/traffic_solution"
os.makedirs(WORK_DIR, exist_ok=True)

OUTPUT_DIR="/content/drive/MyDrive/Barbados/traffic_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Working directory: {WORK_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

config={
    'DATA_DIR':DATA_DIR,
    'TRAIN_CSV':TRAIN_CSV,
    'TEST_CSV':TEST_CSV,
    'WORK_DIR':WORK_DIR,
    'OUTPUT_DIR': OUTPUT_DIR,
    'video_folders':['normanniles1', 'normanniles2', 'normanniles3', 'normanniles4']
}

import json
config_path=os.path.join(WORK_DIR, 'config.json')
with open(config_path, 'w') as f:
  json.dump(config, f, indent=2)

print(f"Configuration saved to: {config_path}")

Working directory: /content/Barbados/traffic_solution
Output directory: /content/drive/MyDrive/Barbados/traffic_output
Configuration saved to: /content/Barbados/traffic_solution/config.json


In [8]:
if TRAIN_CSV and os.path.exists(TRAIN_CSV):
  train_df=pd.read_csv(TRAIN_CSV)

  print("\nTraining data:")
  print(f" -{len(train_df)} samples")
  print(f" -Columns: {', '.join(train_df.columns)}")

  if 'timestamp' in train_df.columns:
    print(f" -Timestamps: {train_df['timestamp'].iloc[0]} to {train_df['timestamp'].iloc[-1]}")

    label_cols=[col for col in train_df.columns if 'congestion' in col.lower() or 'rating' in col.lower()]
    if label_cols:
      print(f"  -Label columns: {', '.join(label_cols)}")
      for col in label_cols:
        print(f"  -{col}: {train_df[col].value_counts().to_dict()}")

  print("\nVideo Folders:")
  for folder_name in ['normanniles1', 'normanniles2', 'normanniles3', 'normanniles4']:
    folder_path=os.path.join(DATA_DIR, folder_name)
    if os.path.exists(folder_path):
      videos=[f for f in os.listdir(folder_path) if f.endswith('.mp4')]
      print(f" -{folder_name}: {len(videos)} videos")
      if videos:
        print(f"  Example: {videos[0]}")



Training data:
 -16076 samples
 -Columns: responseId, view_label, ID_enter, ID_exit, videos, video_time, datetimestamp_start, datetimestamp_end, date, signaling, congestion_enter_rating, congestion_exit_rating, time_segment_id, cycle_phase

Video Folders:
 -normanniles1: 4633 videos
  Example: normanniles1_2025-10-25-12-34-45.mp4
 -normanniles2: 4633 videos
  Example: normanniles2_2025-10-25-11-53-45.mp4
 -normanniles3: 4633 videos
  Example: normanniles3_2025-10-25-11-55-45.mp4
 -normanniles4: 4633 videos
  Example: normanniles4_2025-10-25-11-54-45.mp4


In [1]:
import os
import json
import pandas as pd
from pathlib import Path

In [9]:
WORK_DIR="/content/Barbados/traffic_solution"
config_path=os.path.join(WORK_DIR, 'config.json')

if os.path.exists(config_path):
  with open(config_path, 'r') as f:
    config=json.load(f)

  DATA_DIR=config['DATA_DIR']
  TRAIN_CSV=config['TRAIN_CSV']
  TEST_CSV=config['TEST_CSV']
  OUTPUT_DIR=config['OUTPUT_DIR']



In [10]:
train_df=pd.read_csv(TRAIN_CSV)
print(f"Loaded {len(train_df)} training samples")
print(f"Columns: {list(train_df.columns)}")

print("nFirst 5 rows:")
print(train_df.head())

print("\nMissing values:")
print(train_df.isnull().sum())

label_columns=[col for col in train_df.columns if 'congestion' in col.lower()]
if label_columns:
  print(f"\nLabel columns found: {label_columns}")
  for col in label_columns:
    print(f"\n{col} distribution:")
    print(train_df[col].value_counts())

else:
  print("\nNo 'congestion' columns found. Checking all columns:")
  for col in train_df.columns:
    print(f" - {col}: {train_df[col].dtype}")

Loaded 16076 training samples
Columns: ['responseId', 'view_label', 'ID_enter', 'ID_exit', 'videos', 'video_time', 'datetimestamp_start', 'datetimestamp_end', 'date', 'signaling', 'congestion_enter_rating', 'congestion_exit_rating', 'time_segment_id', 'cycle_phase']
nFirst 5 rows:
                responseId       view_label  \
0  zYkHaeOdB7XOnvgP3YW5kQs  Norman Niles #1   
1  NYsHaeCRLq-vnvgPjoXZqA0  Norman Niles #1   
2  A40HaYT8KNm7nvgPq8e12AU  Norman Niles #1   
3  EIsHaanDMK-vnvgPjoXZqA0  Norman Niles #1   
4  RYsHafSeMaqpmecP5vCV0AQ  Norman Niles #1   

                                            ID_enter  \
0  time_segment_0_Norman Niles #1_congestion_ente...   
1  time_segment_1_Norman Niles #1_congestion_ente...   
2  time_segment_2_Norman Niles #1_congestion_ente...   
3  time_segment_3_Norman Niles #1_congestion_ente...   
4  time_segment_4_Norman Niles #1_congestion_ente...   

                                             ID_exit  \
0  time_segment_0_Norman Niles #1_congesti

In [13]:
def find_video_for_timestamp(timestamp, camera_num, data_dir):
  video_folder=os.path.join(data_dir, f'normanniles{camera_num}')

  if not os.path.exists(video_folder):
    return None

  possible_names=[
      f"{timestamp}_camera{camera_num}.mp4",
      f"{timestamp}.mp4",
      f"camera{camera_num}_{timestamp}.mp4"
  ]

  for name in possible_names:
    video_path=os.path.join(video_folder, name)
    if os.path.exists(video_path):
      return video_path

  all_videos=[f for f in os.listdir(video_folder) if f.endswith('.mp4')]
  for video in all_videos:
    if str(timestamp) in video:
      if str(timestamp) in video:
        return os.path.join(video_folder, video)

  return None

print("Checking video availability...")

if 'timestamp' in train_df.columns:
  timestamp_col='timestamp'
elif 'Timestamp' in train_df.columns:
  timestamp_col='Timestamp'
else:
  timestamp_col=train_df.columns[0]
  print(f"Using '{timestamp_col}' as timestamp column")

video_check=[]
for idx in range(min(5, len(train_df))):
  timestamp=train_df[timestamp_col].iloc[idx]

  videos_found=[]
  for cam in [1, 2, 3, 4]:
    video_path=find_video_for_timestamp(timestamp, cam, DATA_DIR)
    if video_path:
      videos_found.append(f"camera{cam}")

  video_check.append({
      'timestamp':timestamp,
      'videos_found': ', '.join(videos_found) if videos_found else 'NONE',
      'count': len(videos_found)
  })

  print(f" {timestamp}: {len(videos_found)}/4 cameras found")

video_check_df=pd.DataFrame(video_check)

if video_check_df['count'].sum()==0:
  print("\nNo videos found! Check naming convention")
  print("\nLet's inspect video file names:")

  for cam in [1, 2, 3, 4]:
    folder=os.path.join(DATA_DIR, f'normanniles{cam}')
    if os.path.exists(folder):
      videos=[f for f in os.listdir(folder) if f.endswith('.mp4')]
      if videos:
        print(f"\nSample from normanniles{cam}:")
        for v in videos[:3]:
          print(f" -{v}")
else:
  print(f"\nVideos found for training data")

Checking video availability...
Using 'responseId' as timestamp column
 zYkHaeOdB7XOnvgP3YW5kQs: 0/4 cameras found
 NYsHaeCRLq-vnvgPjoXZqA0: 0/4 cameras found
 A40HaYT8KNm7nvgPq8e12AU: 0/4 cameras found
 EIsHaanDMK-vnvgPjoXZqA0: 0/4 cameras found
 RYsHafSeMaqpmecP5vCV0AQ: 0/4 cameras found

No videos found! Check naming convention

Let's inspect video file names:

Sample from normanniles1:
 -normanniles1_2025-10-25-12-34-45.mp4
 -normanniles1_2025-10-25-12-32-45.mp4
 -normanniles1_2025-10-25-12-39-45.mp4

Sample from normanniles2:
 -normanniles2_2025-10-25-11-53-45.mp4
 -normanniles2_2025-10-25-11-56-45.mp4
 -normanniles2_2025-10-25-11-54-45.mp4

Sample from normanniles3:
 -normanniles3_2025-10-25-11-55-45.mp4
 -normanniles3_2025-10-25-11-54-45.mp4
 -normanniles3_2025-10-25-11-56-45.mp4

Sample from normanniles4:
 -normanniles4_2025-10-25-11-54-45.mp4
 -normanniles4_2025-10-25-11-56-45.mp4
 -normanniles4_2025-10-25-11-55-45.mp4
